# 05 — Deployment

Deployment ligero (no productivo, no es un servicio): carga el modelo
registrado en el Model Registry (alias `production`) y genera un ranking de
prioridad de mantenimiento sobre la medición más reciente de cada ruta.

Esto mismo se puede correr desde terminal, fuera del notebook, con:

```bash
python -m src.scoring --output alerts.csv --top 20
```


In [ ]:
import sys
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

import sys
sys.path.insert(0, str(Path("..").resolve() / "src"))
from scoring import score_latest_measurements


In [ ]:
ranking = score_latest_measurements(
    model_name="otdr_risk_classifier",
    alias="production",
    features_path=Path("../data/processed/features_by_route_timeline.parquet"),
)
ranking.head(20)


In [ ]:
out_path = Path("../reports/alerts.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
ranking.to_csv(out_path, index=False)
print(f"[+] Ranking completo guardado en: {out_path}")


## Uso recurrente

Para correr este scoring periódicamente (ej. diario, vía cron) sin abrir el
notebook, usar directamente el script `src/scoring.py` desde la raíz del repo:

```bash
source .venv/bin/activate
python -m src.scoring --model-name otdr_risk_classifier --alias production \
    --output reports/alerts.csv --top 20
```

Antes de eso, hay que refrescar `data/processed/` corriendo de nuevo
`02_data_preparation.ipynb` con datos extraídos más recientes del FMS, y
re-entrenar/registrar el modelo (`03_modeling.ipynb` + `04_evaluation.ipynb`)
si se detecta degradación del modelo en producción.
